In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


No GPU available. Training will run on CPU.


In [3]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4

In [5]:
w_q = nn.Linear(d_model, d_model)
w_k = nn.Linear(d_model, d_model)
w_v = nn.Linear(d_model, d_model)
w_o = nn.Linear(d_model, d_model)

In [4]:
dummy_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.1641, 0.4547, 0.2812, 0.0417, 0.9909, 0.4539, 0.2534, 0.3314],
         [0.5509, 0.0622, 0.7468, 0.0689, 0.3039, 0.9009, 0.8023, 0.6821],
         [0.9992, 0.7589, 0.9056, 0.2315, 0.3275, 0.0815, 0.7714, 0.0604]],

        [[0.7673, 0.4352, 0.4585, 0.0510, 0.6191, 0.7255, 0.8540, 0.2227],
         [0.6173, 0.9239, 0.6051, 0.1757, 0.1383, 0.6399, 0.7347, 0.4492],
         [0.5383, 0.0822, 0.3278, 0.8035, 0.3838, 0.4859, 0.9674, 0.1880]]])

In [6]:
q = w_q(dummy_input)
k = w_k(dummy_input)
v = w_v(dummy_input)

In [7]:
print(q.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 8])
(batch_size, num_tokens, key_dimension)


tensor([[[ 0.0637,  0.4653,  0.6794, -0.2872,  0.1251,  0.5068, -0.3009,
           0.0768],
         [-0.6444,  0.2473,  0.5746, -0.1076, -0.1404,  0.1126, -0.2085,
           0.0965],
         [-0.1744,  0.1404,  0.3355, -0.1985, -0.0792,  0.0825, -0.1618,
           0.3054]],

        [[-0.3608,  0.3718,  0.6471, -0.0533,  0.0436,  0.2968, -0.4361,
          -0.0675],
         [-0.3405,  0.4191,  0.5523, -0.1837, -0.0441,  0.3290, -0.3735,
           0.2515],
         [-0.1113,  0.4702,  0.6355, -0.1450, -0.0030,  0.3081, -0.5133,
           0.0116]]], grad_fn=<ViewBackward0>)

In [30]:
q_head = q.view(batch_size, num_tokens, num_heads, head_dim)
k_head = k.view(batch_size, num_tokens, num_heads, head_dim)
v_head = v.view(batch_size, num_tokens, num_heads, head_dim)
print(q_head.shape)
q_head

torch.Size([2, 3, 2, 4])


tensor([[[[ 0.0637,  0.4653,  0.6794, -0.2872],
          [ 0.1251,  0.5068, -0.3009,  0.0768]],

         [[-0.6444,  0.2473,  0.5746, -0.1076],
          [-0.1404,  0.1126, -0.2085,  0.0965]],

         [[-0.1744,  0.1404,  0.3355, -0.1985],
          [-0.0792,  0.0825, -0.1618,  0.3054]]],


        [[[-0.3608,  0.3718,  0.6471, -0.0533],
          [ 0.0436,  0.2968, -0.4361, -0.0675]],

         [[-0.3405,  0.4191,  0.5523, -0.1837],
          [-0.0441,  0.3290, -0.3735,  0.2515]],

         [[-0.1113,  0.4702,  0.6355, -0.1450],
          [-0.0030,  0.3081, -0.5133,  0.0116]]]], grad_fn=<ViewBackward0>)

In [31]:
q_head = q_head.transpose(1, 2)
k_head = k_head.transpose(1, 2)
v_head = v_head.transpose(1, 2)
print(q_head.shape)
print("(batch_size, num_heads, num_tokens, head_dim)")
q_head

torch.Size([2, 2, 3, 4])
(batch_size, num_heads, num_tokens, head_dim)


tensor([[[[ 0.0637,  0.4653,  0.6794, -0.2872],
          [-0.6444,  0.2473,  0.5746, -0.1076],
          [-0.1744,  0.1404,  0.3355, -0.1985]],

         [[ 0.1251,  0.5068, -0.3009,  0.0768],
          [-0.1404,  0.1126, -0.2085,  0.0965],
          [-0.0792,  0.0825, -0.1618,  0.3054]]],


        [[[-0.3608,  0.3718,  0.6471, -0.0533],
          [-0.3405,  0.4191,  0.5523, -0.1837],
          [-0.1113,  0.4702,  0.6355, -0.1450]],

         [[ 0.0436,  0.2968, -0.4361, -0.0675],
          [-0.0441,  0.3290, -0.3735,  0.2515],
          [-0.0030,  0.3081, -0.5133,  0.0116]]]],
       grad_fn=<TransposeBackward0>)

In [32]:
k_t = k_head.transpose(-2, -1)
print(k_t.shape)
k_t

torch.Size([2, 2, 4, 3])


tensor([[[[ 6.6958e-02, -1.5930e-01, -6.4594e-01],
          [-7.7996e-02,  7.4680e-02, -1.0080e-01],
          [ 1.3911e-01,  2.6541e-01,  1.1688e-01],
          [ 6.5020e-02, -1.7564e-01, -3.4217e-01]],

         [[ 4.4299e-01, -7.1435e-02,  4.5901e-01],
          [ 3.9245e-01,  4.7138e-01,  5.8615e-01],
          [-8.0462e-02, -2.4568e-01,  9.2810e-02],
          [ 1.6349e-01,  9.7306e-02, -3.0497e-01]]],


        [[[-1.1588e-01, -3.2447e-01, -5.0733e-01],
          [-1.3613e-01, -2.0178e-01,  3.0922e-03],
          [ 2.6850e-01,  1.3395e-01,  5.7593e-02],
          [-6.7634e-02, -1.9949e-01, -2.5584e-01]],

         [[ 3.6437e-01,  2.3380e-01,  1.8229e-01],
          [ 5.2431e-01,  3.4459e-01,  6.6638e-01],
          [ 7.2995e-03,  6.9991e-05, -2.9678e-01],
          [-2.2119e-02, -4.7770e-02, -8.3750e-02]]]],
       grad_fn=<TransposeBackward0>)

In [34]:
sim1 = (q_head @ k_t)/math.sqrt(head_dim)
print(sim1.shape)
print("(batch, num_heads, num_queries, num_keys")
sim1
# Attention Scores Matrix

torch.Size([2, 2, 3, 3])
(batch, num_heads, num_queries, num_keys


tensor([[[[ 0.0219,  0.1277,  0.0448],
          [ 0.0053,  0.1463,  0.2477],
          [ 0.0056,  0.0811,  0.1028]],

         [[ 0.1455,  0.1557,  0.1516],
          [ 0.0073,  0.0619, -0.0236],
          [ 0.0301,  0.0570, -0.0481]]],


        [[[ 0.0843,  0.0697,  0.1176],
          [ 0.0716,  0.0683,  0.1264],
          [ 0.0647,  0.0276,  0.0658]],

         [[ 0.0849,  0.0578,  0.1704],
          [ 0.0741,  0.0455,  0.1505],
          [ 0.0782,  0.0524,  0.1781]]]], grad_fn=<DivBackward0>)

In [40]:
sim2 = F.softmax(sim1, dim=-1)
print(sim2.shape)
print("(num_batches, num_heads, num_queries, num_keys)")
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

torch.Size([2, 2, 3, 3])
(num_batches, num_heads, num_queries, num_keys)


tensor([[[[0.3190, 0.3546, 0.3264],
          [0.2919, 0.3361, 0.3720],
          [0.3144, 0.3391, 0.3465]],

         [[0.3315, 0.3349, 0.3335],
          [0.3305, 0.3490, 0.3205],
          [0.3387, 0.3480, 0.3133]]],


        [[[0.3312, 0.3264, 0.3424],
          [0.3275, 0.3265, 0.3460],
          [0.3373, 0.3250, 0.3377]],

         [[0.3265, 0.3178, 0.3557],
          [0.3277, 0.3185, 0.3538],
          [0.3247, 0.3165, 0.3588]]]], grad_fn=<SoftmaxBackward0>)

In [42]:
print(v_head.shape)
v_head # Value vectors

torch.Size([2, 2, 3, 4])


tensor([[[[-0.2584,  0.2359, -0.2184, -0.5517],
          [ 0.3617,  0.1351,  0.1894, -0.2387],
          [ 0.0353,  0.0697, -0.0165, -0.1913]],

         [[-0.4173, -0.1015, -0.0277, -0.0166],
          [-0.2465,  0.1930,  0.1727, -0.0116],
          [-0.1047, -0.0286,  0.0677,  0.7633]]],


        [[[ 0.1114,  0.1604,  0.1787, -0.3319],
          [-0.0178,  0.0490,  0.0953, -0.1627],
          [-0.0042, -0.0538,  0.2370, -0.3819]],

         [[-0.2513,  0.1566,  0.1334,  0.1944],
          [ 0.0127, -0.0377,  0.2364,  0.3360],
          [ 0.0695,  0.0403, -0.1851,  0.1830]]]],
       grad_fn=<TransposeBackward0>)

In [ ]:
sim3 = sim2 @ v_head
print(sim3.shape)
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

torch.Size([2, 2, 3, 4])


tensor([[[[ 0.0574,  0.1459, -0.0079, -0.3231],
          [ 0.0593,  0.1402, -0.0062, -0.3125],
          [ 0.0536,  0.1441, -0.0102, -0.3207]],

         [[-0.2558,  0.0215,  0.0713,  0.2452],
          [-0.2575,  0.0247,  0.0728,  0.2351],
          [-0.2599,  0.0238,  0.0719,  0.2295]]],


        [[[ 0.0297,  0.0507,  0.1714, -0.2938],
          [ 0.0292,  0.0499,  0.1716, -0.2940],
          [ 0.0304,  0.0519,  0.1713, -0.2938]],

         [[-0.0533,  0.0535,  0.0528,  0.2354],
          [-0.0537,  0.0536,  0.0535,  0.2355],
          [-0.0526,  0.0534,  0.0517,  0.2351]]]],
       grad_fn=<UnsafeViewBackward0>)

In [50]:
sim3 = sim3.transpose(1, 2).contiguous()
sim3 = sim3.view(2, 3, 8)
print(sim3.shape)
sim3

torch.Size([2, 3, 8])


tensor([[[ 0.0574,  0.0536, -0.2575,  0.1459,  0.1441,  0.0247, -0.0079,
          -0.0102],
         [ 0.0728, -0.3231, -0.3207,  0.2351,  0.0593, -0.2558, -0.2599,
           0.1402],
         [ 0.0215,  0.0238, -0.0062,  0.0713,  0.0719, -0.3125,  0.2452,
           0.2295]],

        [[ 0.0297,  0.0304, -0.0537,  0.0507,  0.0519,  0.0536,  0.1714,
           0.1713],
         [ 0.0535, -0.2938, -0.2938,  0.2355,  0.0292, -0.0533, -0.0526,
           0.0499],
         [ 0.0535,  0.0534,  0.1716,  0.0528,  0.0517, -0.2940,  0.2354,
           0.2351]]], grad_fn=<ViewBackward0>)

In [47]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our input. Ready to be passed on to the next layer.

torch.Size([2, 3, 8])


tensor([[[ 0.1718,  0.2371, -0.2975,  0.0663,  0.3184, -0.1685,  0.3879,
           0.3376],
         [ 0.1104,  0.0845, -0.1412,  0.1418,  0.4586, -0.1316,  0.3340,
           0.1839],
         [ 0.0600, -0.0825, -0.2279,  0.1343,  0.3860, -0.2438,  0.1781,
           0.0181]],

        [[ 0.1892,  0.2289, -0.3953, -0.0081,  0.2196, -0.0852,  0.3216,
           0.3020],
         [ 0.2065,  0.1091, -0.2767,  0.0048,  0.4015, -0.0391,  0.3378,
           0.1043],
         [ 0.1650, -0.0898, -0.2087,  0.0880,  0.3284, -0.2198,  0.1887,
           0.0245]]], grad_fn=<ViewBackward0>)

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        self.w_o = nn.Linear(self.d_model, self.d_model)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [4]:
dummy_input = torch.rand(1, 3, 4)
dummy_input

tensor([[[0.4822, 0.5225, 0.2562, 0.3452],
         [0.5920, 0.5045, 0.6581, 0.3271],
         [0.1261, 0.9438, 0.0162, 0.2651]]])